In [1]:
import os
import pandas as pd
from preprocessing_mask_segmentation import preprocessing as pre
from shape_features import shape_features as sf
from split import split
from resize_image import resize as res

In [2]:
# Constant variables
FOLDER_PATH = "../images/"
CSV_PATH = "../labels.csv"
MERGED_CSV_AND_FEATURE_PATH = "../features.csv"

In [3]:
images_names = sorted([f for f in os.listdir(FOLDER_PATH) if f.endswith('.png')])

# Import and preprocess images
images_dict = {}
for image_name in images_names:
    full_path = os.path.join(FOLDER_PATH, image_name)
    key = image_name.split('.')[0]
    img = pre.clean_mask(full_path)
    img = pre.smooth_mask_edges(img)

    # Image Resizing - uncomment if CNN used
    # img_resized = res.resize_with_padding(img)
    images_dict[key] = img

# Extract features from images
features_results = []

for name, img in images_dict.items():
    features = sf.extract_shape_features(img)
    if features:
        features['id'] = name
        features_results.append(features)
    else:
        print(f"{name} ommited (empty mask or unexpected error).")

In [4]:
# Import CSV file and
csv_file = pd.read_csv(CSV_PATH)

# Convert features_results into Pandas DataFrame
df_feature_results = pd.DataFrame(features_results)

# Convert id columns to string to prepare merge
csv_file['id'] = csv_file['id'].astype(str)
df_feature_results["id"] = df_feature_results["id"].astype(str)

# Merge CSV columns and feature DataFrame into one final DataFrame
df_final = pd.merge(csv_file, df_feature_results, left_on='id', right_on='id')

# Drop useless columns and convert target feature into binary value
df_final.drop(columns='abnormality_type', inplace=True)
# df_final['pathology'] = (1 if df_final['pathology'] == "MALIGNANT" else 0)
mapping = {'MALIGNANT': 1, 'BENIGN': 0}
df_final['pathology'] = df_final['pathology'].map(mapping)

# Export final DataFrame into CSV file
df_final.to_csv(MERGED_CSV_AND_FEATURE_PATH, index=False)

## Test of Feature Extraction and Preprocessing

In [5]:
df_final.head()

,id,patient_id,assessment,pathology,Area,Area Bounding Box,Area Convex,Area Filled,Axis Major Length,Axis Minor Length,...,Roundness,Shape Factor,Relative Area,Hu Moment 1,Hu Moment 2,Hu Moment 3,Hu Moment 4,Hu Moment 5,Hu Moment 6,Hu Moment 7
0,1001,P_00001,4,1,120176.0,184775.0,144592.0,120176.0,437.817532,369.842129,...,0.798255,58.486064,0.408762,-9.392190,-17.230556,-21.742257,-20.574172,-41.554963,-29.188242,41.60581
1,1002,P_00001,4,1,32114.0,54432.0,39160.0,32114.0,215.156997,204.049262,...,0.883270,39.090181,0.377776,-8.246705,-13.941255,-17.767327,-17.634853,-35.211028,-24.604329,35.15640
2,1003,P_00004,4,0,113057.0,162306.0,132259.0,113057.0,403.087349,373.343420,...,0.885950,54.001930,0.460630,-9.328990,-16.425357,-21.657411,-19.398799,-39.878430,-27.140005,-39.57749
3,1004,P_00004,4,0,89182.0,147452.0,111392.0,89182.0,361.346738,333.637192,...,0.869639,74.353610,0.390348,-9.129767,-16.061523,-21.006824,-19.481583,-39.669806,-27.408701,-39.40404
4,1005,P_00004,4,0,91546.0,151434.0,108858.0,91546.0,389.640992,311.266317,...,0.767751,49.003800,0.386075,-9.153207,-16.994711,-20.714296,-18.637163,37.810392,26.995989,38.29029


In [6]:
df_final.describe()

,assessment,pathology,Area,Area Bounding Box,Area Convex,Area Filled,Axis Major Length,Axis Minor Length,Centroid X,Centroid Y,...,Roundness,Shape Factor,Relative Area,Hu Moment 1,Hu Moment 2,Hu Moment 3,Hu Moment 4,Hu Moment 5,Hu Moment 6,Hu Moment 7
count,1664.000000,1664.000000,1.664000e+03,1.664000e+03,1.664000e+03,1.664000e+03,1664.000000,1664.000000,1664.000000,1664.000000,...,1664.000000,1664.000000,1664.000000,1664.000000,1664.000000,1664.000000,1664.000000,1664.000000,1664.000000,1664.000000
mean,3.501202,0.458534,7.868357e+04,1.221009e+05,9.073593e+04,7.868357e+04,334.169193,269.915583,206.031864,202.283244,...,0.773771,38.582825,0.405954,-8.767491,-15.997799,-19.860943,-18.173624,-5.668878,-3.651311,0.093411
std,1.404828,0.498427,8.055054e+04,1.206283e+05,9.094821e+04,8.055054e+04,137.835099,113.287892,83.124890,81.275151,...,0.110875,12.377596,0.044404,0.643740,1.371947,1.600641,1.634875,36.608998,25.729495,37.016106
min,0.000000,0.000000,3.966000e+03,6.095000e+03,4.643000e+03,3.966000e+03,80.596272,52.545921,51.516357,53.855270,...,0.202718,13.488374,0.253039,-11.402733,-20.963974,-25.869224,-24.200229,-48.375275,-34.108866,-47.280762
25%,3.000000,0.000000,3.621375e+04,5.792400e+04,4.223800e+04,3.621375e+04,245.383208,195.395222,152.343274,148.652792,...,0.711848,29.952978,0.377459,-9.140233,-16.804562,-20.777895,-19.135385,-37.236350,-26.111950,-36.714913
50%,4.000000,0.000000,5.651000e+04,8.885800e+04,6.547500e+04,5.651000e+04,305.682136,247.529924,188.373131,186.009010,...,0.789073,37.052481,0.406828,-8.735862,-15.946672,-19.804899,-18.118055,-33.381654,-23.326326,26.018202
75%,4.000000,1.000000,9.042775e+04,1.425425e+05,1.045720e+05,9.042775e+04,383.560436,318.783599,237.281471,236.437932,...,0.855228,45.188132,0.434657,-8.353138,-15.078655,-18.843429,-17.098298,36.170605,25.361047,36.752786
max,5.000000,1.000000,1.251936e+06,1.797881e+06,1.345671e+06,1.251936e+06,1317.697863,1222.367729,811.905826,791.568087,...,0.987541,115.080035,0.624698,-6.424796,-11.497612,-13.782675,-12.431511,47.437270,33.471862,49.230771


## Data Split
Perform `patient_id` aware split. Extract X features and y labels into separate DataFrames. Split X and y into train and test sets.
## (NOTE: CHANGE DROP_COLS LIST WHEN ALL NEEDED FEATURES ARE SELECTED)

In [7]:
drop_cols = ['id', 'patient_id', 'pathology'] # DO NOT REMOVE ASSESSMENT
X_train, X_test, y_train, y_test, is_intersect_empty = split.aware_patient_split(df_final, drop_cols)

# Verify that the intersection of patient IDs in Train and Test sets is empty
is_intersect_empty # Result: True

True

## Model Training

Basic model training - for test purposes.

In [8]:
from models import models

lr = models.train_lr(X_train, y_train)
svm = models.train_svm(X_train, y_train)
rf = models.train_rf(X_train, y_train)
knn = models.train_knn(X_train, y_train)

result_lr = models.test_model(lr, X_test, y_test)
result_svm = models.test_model(svm, X_test, y_test)
result_rf = models.test_model(rf, X_test, y_test)
result_knn = models.test_model(knn, X_test, y_test)

C:\Users\KONRAD_ADMIN\Desktop\mammography-delta\.venv\lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## Model Results

Basic model results - for testing purposes

## (NOTE: BECAUSE OF FEATURES CORRELATION AND NO FEATURE SCALLING, RESULTS ARE RANDOM RIGHT NOW)

In [9]:
print(result_lr)

[[150  34]
 [102  55]]


In [10]:
print(result_svm)

[[153  31]
 [102  55]]


In [11]:
print(result_rf)

[[148  36]
 [ 46 111]]


In [12]:
print(result_knn)

[[113  71]
 [ 88  69]]
